# Quantum Search in Graph Nodes - Tutorial 2: Exploring Graph Types

This tutorial explores different graph types and how they affect search algorithms.

## Graph Types Covered
1. **Random Graph**: Erdős-Rényi model
2. **Scale-Free Graph**: Power-law degree distribution (Barabási-Albert)
3. **Small-World Graph**: High clustering with short paths (Watts-Strogatz)

## 1. Graph Type Comparison

In [ ]:
from graphs import create_graph, GraphGenerator
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# Create graphs of each type with same number of nodes
n_nodes = 64
seed = 42

graphs = {
    'Random': create_graph('random', n_nodes=n_nodes, seed=seed),
    'Scale-Free': create_graph('scale_free', n_nodes=n_nodes, seed=seed),
    'Small-World': create_graph('small_world', n_nodes=n_nodes, seed=seed)
}

# Compute statistics
generator = GraphGenerator()
stats = {}

for graph_type, graph in graphs.items():
    graph_stats = generator.get_graph_stats(graph)
    stats[graph_type] = graph_stats

# Display as table
stats_df = pd.DataFrame(stats).T
print(stats_df)
print(f"\nAll graphs have {n_nodes} nodes")

## 2. Visualizing Graph Structures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (graph_type, graph) in enumerate(graphs.items()):
    # Use spring layout for consistent visualization
    pos = nx.spring_layout(graph, k=0.5, iterations=50, seed=42)
    
    ax = axes[idx]
    nx.draw_networkx_nodes(graph, pos, node_color='lightblue', node_size=200, ax=ax)
    nx.draw_networkx_edges(graph, pos, alpha=0.2, ax=ax, width=0.5)
    
    ax.set_title(f"{graph_type} Graph ({n_nodes} nodes)")
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Degree Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (graph_type, graph) in enumerate(graphs.items()):
    # Get degree sequence
    degrees = [degree for node, degree in graph.degree()]
    
    ax = axes[idx]
    ax.hist(degrees, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Degree')
    ax.set_ylabel('Number of Nodes')
    ax.set_title(f"{graph_type} - Degree Distribution")
    ax.grid(axis='y', alpha=0.3)
    
    # Print statistics
    print(f"{graph_type} Graph:")
    print(f"  Min degree: {min(degrees)}")
    print(f"  Max degree: {max(degrees)}")
    print(f"  Avg degree: {sum(degrees)/len(degrees):.2f}")
    print()

plt.tight_layout()
plt.show()

## 4. Path Lengths Analysis

In [ ]:
import numpy as np

path_lengths = {}

for graph_type, graph in graphs.items():
    # Check if graph is connected
    if nx.is_connected(graph):
        # Compute all shortest path lengths
        all_lengths = []
        for source in graph.nodes():
            lengths = nx.single_source_shortest_path_length(graph, source)
            all_lengths.extend(lengths.values())
        
        path_lengths[graph_type] = {
            'min': min(all_lengths),
            'max': max(all_lengths),
            'mean': np.mean(all_lengths),
            'diameter': nx.diameter(graph)
        }
    else:
        print(f"{graph_type} graph is not connected!")

# Display as table
path_df = pd.DataFrame(path_lengths).T
print(path_df)

## 5. Classical Search Performance on Different Graphs

In [ ]:
from classical import run_classical_search

target = 32  # Same target for all graphs

results = []

for graph_type, graph in graphs.items():
    for method in ['linear', 'bfs', 'dfs']:
        result = run_classical_search(graph, target, method=method)
        results.append({
            'Graph Type': graph_type,
            'Method': method.upper(),
            'Nodes Checked': result['nodes_checked'],
            'Time (ms)': result['execution_time'] * 1000,
            'Found': 'Yes' if result['found'] else 'No'
        })

results_df = pd.DataFrame(results)
print(results_df)

# Compare by graph type
print("\nNodes Checked by Graph Type:")
print(results_df.pivot(index='Graph Type', columns='Method', values='Nodes Checked'))

## 6. Visualization: Classical Search Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Nodes checked
pivot_nodes = results_df.pivot(index='Graph Type', columns='Method', values='Nodes Checked')
pivot_nodes.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0].set_ylabel('Nodes Checked')
axes[0].set_title('Classical Search - Nodes Checked')
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(title='Method')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

# Time
pivot_time = results_df.pivot(index='Graph Type', columns='Method', values='Time (ms)')
pivot_time.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c', '#2ecc71'])
axes[1].set_ylabel('Execution Time (ms)')
axes[1].set_title('Classical Search - Execution Time')
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(title='Method')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 7. Quantum Search on Different Graphs

In [ ]:
from quantum import run_grover_search

# Run Grover's algorithm for each graph type
# Note: Quantum search is independent of graph structure - it searches node index space
quantum_results = []

for _ in range(3):  # Run 3 times to show consistency
    for graph_type in graphs.keys():
        result = run_grover_search(n_nodes=n_nodes, target=target, shots=1000)
        quantum_results.append({
            'Graph Type': graph_type,
            'Target': target,
            'Grover Iterations': result['grover_iterations'],
            'Success Prob (%)': result['success_probability'],
            'Time (ms)': result['execution_time'] * 1000,
            'Measured Node': result['measured_node']
        })

quantum_df = pd.DataFrame(quantum_results)
print(quantum_df.head(10))
print(f"\nAverage Success Probability: {quantum_df['Success Prob (%)'].mean():.1f}%")
print(f"Avg Grover Iterations: {quantum_df['Grover Iterations'].mean():.1f}")

## 8. Key Insights

### Random Graph (Erdős-Rényi)
- Uniform degree distribution
- Moderate clustering
- Average path length grows logarithmically

### Scale-Free Graph (Barabási-Albert)
- Power-law degree distribution
- Few hubs, many low-degree nodes
- Very short average path length
- Common in real-world networks (social, biological)

### Small-World Graph (Watts-Strogatz)
- High clustering coefficient
- Short average path length
- Balance between local and global connections
- Models real networks well (collaboration networks, etc.)

## Important Observation
**Quantum search performance is independent of graph structure** - it depends only on the number of nodes (N). The graph structure affects classical BFS/DFS, but Grover's algorithm searches the state space directly.

## Next Steps
- Run comprehensive benchmarks in Tutorial 3
- Analyze quantum circuits in Tutorial 4